In [ ]:
{
 "nbformat": 4,
 "nbformat_minor": 5,
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  },
  "colab": {
   "provenance": [],
   "gpuType": "T4"
  },
  "accelerator": "GPU"
 },
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 📊 Step 3 — Evaluation\n",
    "\n",
    "**Run this notebook after `02_training_colab.ipynb`.**\n",
    "\n",
    "This notebook:\n",
    "1. Loads the fine-tuned model from Google Drive\n",
    "2. Runs automatic metrics — BLEU, ROUGE, Recall@K\n",
    "3. Tests the full RAG pipeline\n",
    "4. Checks for hallucinated section numbers\n",
    "5. Generates a full evaluation report\n",
    "\n",
    "---\n",
    "> T4 GPU is enough for evaluation — no need for A100."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 1 — Mount Drive and Clone Repo"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from google.colab import drive\n",
    "drive.mount('/content/drive')\n",
    "\n",
    "import os\n",
    "import sys\n",
    "\n",
    "GITHUB_USERNAME = 'YOUR_USERNAME'\n",
    "REPO_NAME       = 'kannada-legal-ai'\n",
    "DRIVE_PATH      = '/content/drive/MyDrive/kannada-legal-ai'\n",
    "\n",
    "# Clone repo\n",
    "if not os.path.exists(f'/content/{REPO_NAME}'):\n",
    "    !git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git\n",
    "else:\n",
    "    !cd /content/{REPO_NAME} && git pull\n",
    "\n",
    "%cd /content/{REPO_NAME}\n",
    "sys.path.insert(0, f'/content/{REPO_NAME}')\n",
    "\n",
    "print(f'Working directory: {os.getcwd()}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 2 — Install Dependencies"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!pip install transformers peft bitsandbytes \\\n",
    "             evaluate rouge-score nltk \\\n",
    "             sentence-transformers chromadb \\\n",
    "             rank-bm25 indic-nlp-library \\\n",
    "             loguru sentencepiece -q\n",
    "\n",
    "import nltk\n",
    "nltk.download('punkt', quiet=True)\n",
    "nltk.download('punkt_tab', quiet=True)\n",
    "\n",
    "print('\\n✅ Dependencies installed!')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 3 — Load Data from Drive"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import shutil\n",
    "\n",
    "# Copy data from Drive\n",
    "folders = ['annotated', 'processed', 'vector_store']\n",
    "for folder in folders:\n",
    "    src = f'{DRIVE_PATH}/data/{folder}'\n",
    "    dst = f'data/{folder}'\n",
    "    if os.path.exists(src):\n",
    "        shutil.copytree(src, dst, dirs_exist_ok=True)\n",
    "        print(f'✅ Loaded: data/{folder}')\n",
    "    else:\n",
    "        print(f'⚠️  Not found: {src}')\n",
    "\n",
    "# Copy trained model from Drive\n",
    "src_model = f'{DRIVE_PATH}/best_model'\n",
    "dst_model = 'training/checkpoints/best_model'\n",
    "if os.path.exists(src_model):\n",
    "    os.makedirs('training/checkpoints', exist_ok=True)\n",
    "    shutil.copytree(src_model, dst_model, dirs_exist_ok=True)\n",
    "    print(f'✅ Loaded: trained model')\n",
    "else:\n",
    "    print('⚠️  Trained model not found on Drive.')\n",
    "    print('   Run 02_training_colab.ipynb first.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 4 — Check Test Dataset"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import json\n",
    "\n",
    "test_pairs = []\n",
    "test_path  = 'data/annotated/test/qa_pairs.jsonl'\n",
    "\n",
    "with open(test_path) as f:\n",
    "    for line in f:\n",
    "        test_pairs.append(json.loads(line.strip()))\n",
    "\n",
    "print(f'Test pairs loaded: {len(test_pairs)}')\n",
    "print('\\n── Sample test pair ──')\n",
    "sample = test_pairs[0]\n",
    "print(f'Question : {sample[\"question\"]}')\n",
    "print(f'Answer   : {sample[\"answer\"][:150]}...')\n",
    "print(f'Law      : {sample.get(\"law\", \"?\")} §{sample.get(\"section\", \"?\")}')  "
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 5 — Test RAG Pipeline (Without Fine-tuned Model)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from rag.rag_pipeline import answer\n",
    "\n",
    "print('── RAG Pipeline Test ──\\n')\n",
    "\n",
    "test_queries = [\n",
    "    'IPC ಸೆಕ್ಷನ್ 302 ಏನು?',\n",
    "    'ಪೊಲೀಸ್ ಬಂಧಿಸಿದರೆ ನನ್ನ ಹಕ್ಕೇನು?',\n",
    "    'FIR ದಾಖಲಿಸುವುದು ಹೇಗೆ?',\n",
    "    'ಕಳ್ಳತನಕ್ಕೆ ಎಷ್ಟು ಜೈಲು ಶಿಕ್ಷೆ?',\n",
    "    'ಜಾಮೀನು ಪಡೆಯುವ ಪ್ರಕ್ರಿಯೆ ಏನು?',\n",
    "]\n",
    "\n",
    "rag_results = []\n",
    "\n",
    "for query in test_queries:\n",
    "    result = answer(query, top_k=3)\n",
    "    rag_results.append(result)\n",
    "\n",
    "    print(f'Query   : {query}')\n",
    "    print(f'Intent  : {result[\"intent\"]}')\n",
    "    print(f'Sections: {result[\"section_numbers\"]}')\n",
    "    print(f'Laws    : {result[\"law_names\"]}')\n",
    "\n",
    "    if result['contexts']:\n",
    "        top = result['contexts'][0]\n",
    "        meta = top.get('metadata', {})\n",
    "        print(f'Top ctx : §{meta.get(\"section_number\",\"?\")} '\n",
    "              f'({meta.get(\"law_name\",\"?\")})')\n",
    "        print(f'Text    : {top[\"text\"][:100]}...')\n",
    "    print('-' * 55)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 6 — Load Fine-tuned Model and Generate Answers"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "from transformers import AutoModelForCausalLM, AutoTokenizer\n",
    "from peft import PeftModel\n",
    "\n",
    "MODEL_PATH = 'training/checkpoints/best_model'\n",
    "BASE_MODEL  = 'ai4bharat/indic-bert'\n",
    "\n",
    "print('Loading fine-tuned model...')\n",
    "\n",
    "tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)\n",
    "\n",
    "base_model = AutoModelForCausalLM.from_pretrained(\n",
    "    BASE_MODEL,\n",
    "    torch_dtype=torch.float16,\n",
    "    device_map='auto',\n",
    ")\n",
    "\n",
    "model = PeftModel.from_pretrained(base_model, MODEL_PATH)\n",
    "model.eval()\n",
    "\n",
    "print('✅ Fine-tuned model loaded!')\n",
    "\n",
    "\n",
    "def generate_answer(prompt, max_new_tokens=150):\n",
    "    \"\"\"\n",
    "    Generate a Kannada legal answer from the fine-tuned model.\n",
    "    \"\"\"\n",
    "    inputs = tokenizer(\n",
    "        prompt,\n",
    "        return_tensors='pt',\n",
    "        max_length=512,\n",
    "        truncation=True,\n",
    "    ).to(model.device)\n",
    "\n",
    "    with torch.no_grad():\n",
    "        outputs = model.generate(\n",
    "            **inputs,\n",
    "            max_new_tokens=max_new_tokens,\n",
    "            temperature=0.7,\n",
    "            do_sample=True,\n",
    "            pad_token_id=tokenizer.eos_token_id,\n",
    "            repetition_penalty=1.2,\n",
    "        )\n",
    "\n",
    "    generated = tokenizer.decode(\n",
    "        outputs[0],\n",
    "        skip_special_tokens=True,\n",
    "    )\n",
    "\n",
    "    # Extract answer part\n",
    "    if '### ಉತ್ತರ:' in generated:\n",
    "        answer = generated.split('### ಉತ್ತರ:')[-1].strip()\n",
    "    else:\n",
    "        answer = generated[len(prompt):].strip()\n",
    "\n",
    "    return answer\n",
    "\n",
    "\n",
    "print('\\n── Generation Test ──')\n",
    "test_prompt = (\n",
    "    '### ಪ್ರಶ್ನೆ:\\n'\n",
    "    'IPC ಸೆಕ್ಷನ್ 302 ಏನು?\\n'\n",
    "    '### ಉತ್ತರ:'\n",
    ")\n",
    "result = generate_answer(test_prompt)\n",
    "print(f'Answer: {result}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 7 — Generate Predictions for Full Test Set"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from tqdm import tqdm\n",
    "\n",
    "PROMPT_TEMPLATE = (\n",
    "    '### ಸಂದರ್ಭ:\\n{context}\\n\\n'\n",
    "    '### ಪ್ರಶ್ನೆ:\\n{question}\\n'\n",
    "    '### ಉತ್ತರ:'\n",
    ")\n",
    "\n",
    "predictions = []\n",
    "references  = []\n",
    "\n",
    "print(f'Generating answers for {len(test_pairs)} test pairs...')\n",
    "\n",
    "for pair in tqdm(test_pairs):\n",
    "    # Get RAG context\n",
    "    rag = answer(pair['question'], top_k=3)\n",
    "    context = rag['contexts'][0]['text'] \\\n",
    "              if rag['contexts'] else ''\n",
    "\n",
    "    # Build prompt\n",
    "    prompt = PROMPT_TEMPLATE.format(\n",
    "        context=context[:300],\n",
    "        question=pair['question'],\n",
    "    )\n",
    "\n",
    "    # Generate answer\n",
    "    pred = generate_answer(prompt)\n",
    "    predictions.append(pred)\n",
    "    references.append(pair['answer'])\n",
    "\n",
    "print(f'\\n✅ Generated {len(predictions)} predictions')\n",
    "print('\\nSample prediction:')\n",
    "print(f'Q  : {test_pairs[0][\"question\"]}')\n",
    "print(f'Ref: {references[0][:100]}...')\n",
    "print(f'Pred: {predictions[0][:100]}...')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 8 — Run Automatic Metrics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import evaluate\n",
    "import re\n",
    "\n",
    "rouge = evaluate.load('rouge')\n",
    "bleu  = evaluate.load('bleu')\n",
    "\n",
    "\n",
    "def section_exact_match(pred, ref):\n",
    "    pred_secs = set(re.findall(r'\\d+[A-Z]?', pred))\n",
    "    ref_secs  = set(re.findall(r'\\d+[A-Z]?', ref))\n",
    "    return bool(pred_secs & ref_secs)\n",
    "\n",
    "\n",
    "# ROUGE\n",
    "rouge_scores = rouge.compute(\n",
    "    predictions=predictions,\n",
    "    references=references,\n",
    ")\n",
    "\n",
    "# BLEU\n",
    "bleu_score = bleu.compute(\n",
    "    predictions=predictions,\n",
    "    references=[[r] for r in references],\n",
    ")\n",
    "\n",
    "# Section exact match\n",
    "section_matches = [\n",
    "    section_exact_match(p, r)\n",
    "    for p, r in zip(predictions, references)\n",
    "]\n",
    "sem_score = sum(section_matches) / len(section_matches)\n",
    "\n",
    "print('══════════════════════════════════')\n",
    "print('   Evaluation Results')\n",
    "print('══════════════════════════════════')\n",
    "print(f'  ROUGE-1         : {rouge_scores[\"rouge1\"]:.4f}')\n",
    "print(f'  ROUGE-2         : {rouge_scores[\"rouge2\"]:.4f}')\n",
    "print(f'  ROUGE-L         : {rouge_scores[\"rougeL\"]:.4f}')\n",
    "print(f'  BLEU            : {bleu_score[\"bleu\"]:.4f}')\n",
    "print(f'  Section Match   : {sem_score:.4f}')\n",
    "print('══════════════════════════════════')\n",
    "\n",
    "# Interpretation\n",
    "print('\\nInterpretation:')\n",
    "print(f'  ROUGE-L > 0.3   : {\"✅ Pass\" if rouge_scores[\"rougeL\"] > 0.3 else \"❌ Needs more data\"}')\n",
    "print(f'  BLEU > 0.15     : {\"✅ Pass\" if bleu_score[\"bleu\"] > 0.15 else \"❌ Needs more data\"}')\n",
    "print(f'  Section > 0.6   : {\"✅ Pass\" if sem_score > 0.6 else \"❌ Needs more data\"}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 9 — Hallucination Check"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import re\n",
    "\n",
    "def check_hallucination(prediction, contexts):\n",
    "    \"\"\"\n",
    "    Check if predicted answer cites sections\n",
    "    not present in retrieved context.\n",
    "    \"\"\"\n",
    "    pred_secs = set(\n",
    "        re.findall(r'(?:Section|ವಿಭಾಗ)\\s*(\\d+[A-Z]?)', prediction)\n",
    "    )\n",
    "    ctx_text = ' '.join(\n",
    "        c['text'] for c in contexts\n",
    "    )\n",
    "    ctx_secs = set(\n",
    "        re.findall(r'(?:Section|ವಿಭಾಗ)\\s*(\\d+[A-Z]?)', ctx_text)\n",
    "    )\n",
    "    hallucinated = pred_secs - ctx_secs\n",
    "    return {\n",
    "        'pred_sections':  list(pred_secs),\n",
    "        'ctx_sections':   list(ctx_secs),\n",
    "        'hallucinated':   list(hallucinated),\n",
    "        'is_hallucinated':len(hallucinated) > 0,\n",
    "    }\n",
    "\n",
    "\n",
    "print('── Hallucination Check ──\\n')\n",
    "\n",
    "hallucination_count = 0\n",
    "\n",
    "for i, (pred, pair) in enumerate(zip(predictions, test_pairs)):\n",
    "    rag    = answer(pair['question'], top_k=3)\n",
    "    result = check_hallucination(pred, rag['contexts'])\n",
    "\n",
    "    if result['is_hallucinated']:\n",
    "        hallucination_count += 1\n",
    "        print(f'⚠️  Query [{i+1}]: {pair[\"question\"]}')\n",
    "        print(f'   Hallucinated sections: {result[\"hallucinated\"]}')\n",
    "\n",
    "total = len(predictions)\n",
    "clean = total - hallucination_count\n",
    "\n",
    "print(f'\\nTotal queries     : {total}')\n",
    "print(f'Clean (no halluc) : {clean} ({round(clean/total*100)}%)')\n",
    "print(f'Hallucinated      : {hallucination_count} ({round(hallucination_count/total*100)}%)')\n",
    "\n",
    "if hallucination_count == 0:\n",
    "    print('\\n✅ No hallucinations detected!')\n",
    "elif hallucination_count / total < 0.2:\n",
    "    print('\\n✅ Hallucination rate acceptable (< 20%)')\n",
    "else:\n",
    "    print('\\n⚠️  High hallucination rate. Add more training data.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 10 — Retrieval Quality (Recall@K)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print('── Retrieval Quality — Recall@K ──\\n')\n",
    "\n",
    "recall_scores = {1: [], 3: [], 5: []}\n",
    "\n",
    "for pair in test_pairs:\n",
    "    expected_section = pair.get('section', '')\n",
    "    if not expected_section:\n",
    "        continue\n",
    "\n",
    "    rag = answer(pair['question'], top_k=5)\n",
    "    retrieved_sections = [\n",
    "        c.get('metadata', {}).get('section_number', '')\n",
    "        for c in rag['contexts']\n",
    "    ]\n",
    "\n",
    "    for k in [1, 3, 5]:\n",
    "        top_k_secs = retrieved_sections[:k]\n",
    "        hit = expected_section in top_k_secs\n",
    "        recall_scores[k].append(int(hit))\n",
    "\n",
    "for k in [1, 3, 5]:\n",
    "    if recall_scores[k]:\n",
    "        score = sum(recall_scores[k]) / len(recall_scores[k])\n",
    "        status = '✅' if score > 0.6 else '⚠️ '\n",
    "        print(f'  {status} Recall@{k} : {score:.4f}')\n",
    "\n",
    "print('\\nRecall@K measures whether the correct')\n",
    "print('legal section was retrieved in top K results.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 11 — Per-Intent Evaluation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from collections import defaultdict\n",
    "\n",
    "intent_scores = defaultdict(list)\n",
    "\n",
    "for pred, pair in zip(predictions, test_pairs):\n",
    "    intent = pair.get('intent', 'general')\n",
    "    match  = section_exact_match(pred, pair['answer'])\n",
    "    intent_scores[intent].append(int(match))\n",
    "\n",
    "print('── Per-Intent Section Match ──\\n')\n",
    "for intent, scores in sorted(intent_scores.items()):\n",
    "    avg    = sum(scores) / len(scores) if scores else 0\n",
    "    status = '✅' if avg > 0.6 else '⚠️ '\n",
    "    print(f'  {status} {intent:20} : {avg:.2f} ({len(scores)} samples)')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 12 — Full Evaluation Report"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import json\n",
    "from datetime import datetime\n",
    "\n",
    "report = {\n",
    "    'timestamp'        : datetime.now().isoformat(),\n",
    "    'model'            : 'ai4bharat/indic-bert + QLoRA',\n",
    "    'test_samples'     : len(test_pairs),\n",
    "    'metrics': {\n",
    "        'rouge1'         : round(rouge_scores['rouge1'], 4),\n",
    "        'rouge2'         : round(rouge_scores['rouge2'], 4),\n",
    "        'rougeL'         : round(rouge_scores['rougeL'], 4),\n",
    "        'bleu'           : round(bleu_score['bleu'], 4),\n",
    "        'section_match'  : round(sem_score, 4),\n",
    "        'hallucination_rate': round(\n",
    "            hallucination_count / len(predictions), 4\n",
    "        ),\n",
    "    },\n",
    "    'per_intent': {\n",
    "        k: round(sum(v)/len(v), 4)\n",
    "        for k, v in intent_scores.items()\n",
    "    },\n",
    "    'sample_predictions': [\n",
    "        {\n",
    "            'question'  : test_pairs[i]['question'],\n",
    "            'reference' : test_pairs[i]['answer'],\n",
    "            'prediction': predictions[i],\n",
    "        }\n",
    "        for i in range(min(3, len(predictions)))\n",
    "    ],\n",
    "}\n",
    "\n",
    "# Save report\n",
    "os.makedirs('evaluation', exist_ok=True)\n",
    "report_path = 'evaluation/eval_report.json'\n",
    "with open(report_path, 'w', encoding='utf-8') as f:\n",
    "    json.dump(report, f, ensure_ascii=False, indent=2)\n",
    "\n",
    "print('══════════════════════════════════════')\n",
    "print('   Final Evaluation Report')\n",
    "print('══════════════════════════════════════')\n",
    "print(f'  Timestamp        : {report[\"timestamp\"]}')\n",
    "print(f'  Model            : {report[\"model\"]}')\n",
    "print(f'  Test samples     : {report[\"test_samples\"]}')\n",
    "print('  ── Metrics ──')\n",
    "for k, v in report['metrics'].items():\n",
    "    print(f'  {k:25} : {v}')\n",
    "print(f'\\n  Report saved to  : {report_path}')\n",
    "print('══════════════════════════════════════')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 13 — Save Report to Google Drive"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import shutil\n",
    "\n",
    "drive_eval = f'{DRIVE_PATH}/evaluation'\n",
    "os.makedirs(drive_eval, exist_ok=True)\n",
    "\n",
    "shutil.copy(\n",
    "    'evaluation/eval_report.json',\n",
    "    f'{drive_eval}/eval_report.json'\n",
    ")\n",
    "\n",
    "print(f'✅ Report saved to Google Drive!')\n",
    "print(f'   {drive_eval}/eval_report.json')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cell 14 — Interactive Test (Try Your Own Queries)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def ask(question):\n",
    "    \"\"\"\n",
    "    Ask a Kannada legal question and get an answer.\n",
    "    Uses RAG + fine-tuned model.\n",
    "    \"\"\"\n",
    "    print(f'\\n❓ {question}')\n",
    "    print('─' * 50)\n",
    "\n",
    "    # RAG retrieval\n",
    "    rag = answer(question, top_k=3)\n",
    "\n",
    "    # Build prompt\n",
    "    context = rag['contexts'][0]['text'] \\\n",
    "              if rag['contexts'] else ''\n",
    "    meta    = rag['contexts'][0].get('metadata', {}) \\\n",
    "              if rag['contexts'] else {}\n",
    "\n",
    "    prompt = (\n",
    "        f'### ಸಂದರ್ಭ:\\n{context[:300]}\\n\\n'\n",
    "        f'### ಪ್ರಶ್ನೆ:\\n{question}\\n'\n",
    "        f'### ಉತ್ತರ:'\n",
    "    )\n",
    "\n",
    "    # Generate\n",
    "    pred = generate_answer(prompt)\n",
    "\n",
    "    print(f'Intent   : {rag[\"intent\"]}')\n",
    "    print(f'Sections : {rag[\"section_numbers\"]}')\n",
    "    print(f'Source   : {meta.get(\"law_name\",\"?\")} §{meta.get(\"section_number\",\"?\")}')\n",
    "    print(f'\\n📖 Answer:\\n{pred}')\n",
    "    print('\\n⚠️  ಇದು ಕಾನೂನು ಸಲಹೆ ಅಲ್ಲ.')\n",
    "\n",
    "\n",
    "# ── Try these queries ──\n",
    "ask('IPC ಸೆಕ್ಷನ್ 302 ಏನು?')\n",
    "ask('ಪೊಲೀಸ್ ಬಂಧಿಸಿದರೆ ನನ್ನ ಹಕ್ಕೇನು?')\n",
    "ask('FIR ದಾಖಲಿಸುವುದು ಹೇಗೆ?')"
   ]
  }
 ]
}